# Lab 6: Get records and a paper from OpenAlex

**Allow 30–40 minutes.** Work in a copy named `lab06_openalex_api_acquisition_mine.ipynb`.

You will use an API key, make a small table, and download one paper. The request code is already written. Your main coding task is to make the table.


## 1. Load your API key · 5 minutes

Get a free API key from your account settings at [OpenAlex](https://openalex.org).

In the course repository's top folder, copy `.env.example` to `.env` if you have not done this yet. Open `.env` and put your key after `OPENALEX_API_KEY=`. Save it.

Keep the key in `.env`, which Git ignores. Do not paste it into this notebook or an AI chat.

Run the cell below. It should print `Key loaded.`


In [ ]:
from pathlib import Path
import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv

for folder in (Path.cwd(), *Path.cwd().parents):
    env_path = folder / ".env"
    if env_path.is_file():
        load_dotenv(env_path, override=True)
        break

key = os.environ.get("OPENALEX_API_KEY", "")
if not key:
    raise ValueError("Add OPENALEX_API_KEY to your .env file, then run this cell again.")

headers = {"Authorization": "Bearer " + key}
print("Key loaded.")


## 2. Get paper records · 5 minutes

This request finds 25 highly cited works from 2024 with a Berkeley-affiliated author **and an available PDF**.

Run the code. You can change the year in `filter` if you want. Status 200 means the request worked. The response is saved so you can reuse it without another request.


In [ ]:
params = {
    "filter": "institutions.id:I95457486,publication_year:2024,has_content.pdf:true",
    "sort": "cited_by_count:desc",
    "per_page": 25,
}

response = requests.get(
    "https://api.openalex.org/works",
    params=params, headers=headers, timeout=30
)
print("Status:", response.status_code)

if response.status_code == 200:
    data = response.json()
    with open("openalex_mine.json", "w") as f:
        json.dump(data, f)
    print("Works received:", len(data["results"]))
else:
    print("The request failed. Use the saved example below.")


**If the request fails:** run this instead. It loads five saved Berkeley records. You can still make the table, but these records do not include PDF download links.

```python
with open("../docs/interactives/data/week06/openalex_page1.json") as f:
    data = json.load(f)
```

If you already saved your own response, load `openalex_mine.json` instead.


## 3. Make a table · 10–15 minutes

Each item in `data["results"]` is a dictionary about one paper.

Use a loop to build a list called `rows`. Add one dictionary per paper with these columns:

| Column | Read from each `work` |
| --- | --- |
| `title` | `work["display_name"]` |
| `cited_by` | `work["cited_by_count"]` |
| `field` | `work["primary_topic"]["field"]["display_name"]` |

If `work["primary_topic"]` is `None`, use `None` for the field.

Make a DataFrame from `rows`. Display it, then count the fields with `table["field"].value_counts()`.


In [ ]:
rows = []

# Loop over data["results"] and add one dictionary to rows per paper.


# Make a DataFrame called table. Display it and count the fields.


## 4. Download one paper · 5 minutes

Our table contains **metadata**: information about papers. Now get the paper itself.

Run this cell to save the first paper as `openalex_paper_mine.pdf`. Open it from your notebook folder and check that its title matches the record.

A PDF is a file, so we save `response.content` rather than use `.json()`. The download URL contains your key; do not print it.


In [ ]:
paper = data["results"][0]
print("Paper:", paper["display_name"])
pdf_url = (paper.get("content_urls") or {}).get("pdf")

if not pdf_url:
    print("No PDF link in this record. Ask for help with the live request.")
else:
    try:
        pdf_response = requests.get(
            pdf_url, params={"api_key": key}, timeout=60
        )
    except requests.RequestException:
        print("The download failed. Try again later.")
    else:
        print("Status:", pdf_response.status_code)
        if pdf_response.status_code == 200 and pdf_response.content.startswith(b"%PDF-"):
            pdf_path = Path("openalex_paper_mine.pdf")
            pdf_path.write_bytes(pdf_response.content)
            print("Saved:", pdf_path.name)
        else:
            print("No PDF was saved. Check your key and daily allowance.")


## 5. What did you learn? · 5 minutes

Write one short sentence for each question.

1. What is one thing you learned from the PDF that is not in the table?
2. Why is this sample not a list of all Berkeley papers?
3. Does an author’s Berkeley affiliation prove they are a faculty member?


Your answers:

## Before you finish

Check that your key does not appear in any cell or output. Do not display `key`, `headers`, or `pdf_response.url`. If you expose the key, replace it in your OpenAlex account settings.

Keep your notebook, saved JSON, and PDF. Your personal files are ignored by Git.

Reserve the last 10 minutes of the lab session for the quiz on Monday's material.
